# Capítulo 2 · Modelado de lenguaje

Un generador de nombres que aprende contando qué letra sigue a cada letra. Python estándar y CPU: no hay que instalar nada. Ejecuta las celdas en orden.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio: el mismo programa, por partes, para ejecutarlo y experimentar.

## 1. Los datos

Cuatro nombres de entrenamiento, dos de reserva que no se cuentan y dos marcas: `^` para el inicio y `$` para el final. Las salidas posibles son las cuatro letras y el final.

In [1]:
"""Capítulo 2: bigramas de caracteres. Solo biblioteca estándar y CPU."""
from collections import Counter
import math
import random

ENTRENAMIENTO = ("ana", "anna", "alba", "alan")
RESERVA = ("lana", "bala")
INICIO, FIN = "^", "$"
LETRAS = tuple(sorted(set("".join(ENTRENAMIENTO))))
SALIDAS = LETRAS + (FIN,)
CONTEXTOS = (INICIO,) + LETRAS

In [2]:
print("Salidas:", SALIDAS)
print("Contextos:", CONTEXTOS)

Salidas: ('a', 'b', 'l', 'n', '$')
Contextos: ('^', 'a', 'b', 'l', 'n')


## 2. Contar

`pares` recorre un nombre por parejas, con sus marcas; `contar` suma uno en la casilla de cada pareja. Antes de ejecutar, predice la fila de `a` y el número total de transiciones.

In [3]:
def pares(nombre):
    cadena = INICIO + nombre + FIN
    return zip(cadena, cadena[1:])


def contar(nombres):
    cuentas = {c: Counter() for c in CONTEXTOS}
    for nombre in nombres:
        if not nombre or any(c not in LETRAS for c in nombre):
            raise ValueError("Nombre vacío o letra fuera del vocabulario")
        for anterior, siguiente in pares(nombre):
            cuentas[anterior][siguiente] += 1
    return cuentas

In [4]:
print("Parejas de ana:", list(pares("ana")))
cuentas = contar(ENTRENAMIENTO)
for c in CONTEXTOS:
    print(c, [cuentas[c][s] for s in SALIDAS])
print("Total:", sum(sum(fila.values()) for fila in cuentas.values()), "transiciones")

Parejas de ana: [('^', 'a'), ('a', 'n'), ('n', 'a'), ('a', '$')]
^ [4, 0, 0, 0, 0]
a [0, 0, 2, 3, 3]
b [1, 0, 0, 0, 0]
l [1, 1, 0, 0, 0]
n [2, 0, 0, 1, 1]
Total: 19 transiciones


## 3. De recuentos a probabilidades

`probabilidades` divide cada fila entre su total. Con `alpha` añade una papeleta imaginaria de cada salida: es el suavizado. ¿Por qué hay trece papeletas después de `a`?

In [5]:
def probabilidades(cuentas, alpha=1.0):
    if alpha < 0:
        raise ValueError("alpha debe ser no negativo")
    tabla = {}
    for c in CONTEXTOS:
        total = sum(cuentas[c].values()) + alpha * len(SALIDAS)
        if total == 0:
            raise ValueError("Fila vacía: utiliza suavizado")
        tabla[c] = {s: (cuentas[c][s] + alpha) / total
                    for s in SALIDAS}
    return tabla

In [6]:
sin_suavizar = probabilidades(cuentas, alpha=0)
tabla = probabilidades(cuentas, alpha=1)
print("Fila de a sin suavizar:", [round(sin_suavizar["a"][s], 3) for s in SALIDAS])
print("Fila de a suavizada:   ", [round(tabla["a"][s], 3) for s in SALIDAS])
print("La fila suavizada suma:", sum(tabla["a"].values()))

Fila de a sin suavizar: [0.0, 0.0, 0.25, 0.375, 0.375]
Fila de a suavizada:    [0.077, 0.077, 0.231, 0.308, 0.308]
La fila suavizada suma: 1.0


## 4. Del coste a la pérdida

El coste de una transición es −ln p: cero si el modelo daba probabilidad 1 a lo que apareció, y más grande cuanto menos probable lo consideraba. Python escribe el primer coste como `-0.0`, un cero con signo.

In [7]:
import math

for p in (1.0, 0.5, 0.1):
    coste = -math.log(p)
    print(p, round(coste, 3))

1.0 -0.0
0.5 0.693
0.1 2.303


`perdida` hace la media de los costes de todas las transiciones de unos textos dados. No sortea nada: la salida ya viene dada por el texto.

In [8]:
def perdida(tabla, nombres):
    costes = []
    for nombre in nombres:
        if not nombre or any(c not in LETRAS for c in nombre):
            raise ValueError("Nombre vacío o letra fuera del vocabulario")
        for c, s in pares(nombre):
            p = tabla[c][s]
            if p == 0:
                return math.inf
            costes.append(-math.log(p))
    if not costes:
        raise ValueError("No hay transiciones para evaluar")
    return sum(costes) / len(costes)

Sigue el camino de `ana` y el de `lana`: la probabilidad del camino es el producto de sus transiciones, y la NLL, la media de sus costes.

In [9]:
for nombre in ("ana", "lana"):
    ps = [tabla[c][s] for c, s in pares(nombre)]
    print(nombre, [round(p, 3) for p in ps])
    print("   probabilidad del camino:", round(math.prod(ps), 6), " NLL media:", round(perdida(tabla, [nombre]), 6))

ana [0.556, 0.308, 0.333, 0.308]
   probabilidad del camino: 0.017532  NLL media: 1.010927
lana [0.111, 0.286, 0.308, 0.333, 0.308]
   probabilidad del camino: 0.001002  NLL media: 1.381182


## 5. Generar

`generar` sortea una salida de la fila actual, la añade y la usa como nuevo contexto, hasta sacar el final o agotar veinte decisiones.

In [10]:
def generar(tabla, rng, max_pasos=20):
    """Devuelve (texto, terminó con FIN). El límite no fuerza un FIN."""
    c, letras = INICIO, []
    for _ in range(max_pasos):
        pesos = [tabla[c][s] for s in SALIDAS]
        s = rng.choices(SALIDAS, weights=pesos, k=1)[0]
        if s == FIN:
            return "".join(letras), True
        letras.append(s)
        c = s
    return "".join(letras), False

## 6. El experimento completo

`ejecutar` cuenta, compara con el modelo uniforme, evalúa con y sin suavizado y genera diez muestras con la semilla 7.

In [11]:
def ejecutar():
    cuentas = contar(ENTRENAMIENTO)
    tabla = probabilidades(cuentas, alpha=1.0)
    uniforme = {c: {s: 1 / len(SALIDAS) for s in SALIDAS}
                for c in CONTEXTOS}
    print("Columnas:", SALIDAS)
    for c in CONTEXTOS:
        print(c, [cuentas[c][s] for s in SALIDAS])
    print("NLL uniforme: %.6f" % perdida(uniforme, RESERVA))
    print("NLL entrenamiento: %.6f" % perdida(tabla, ENTRENAMIENTO))
    print("NLL reserva: %.6f" % perdida(tabla, RESERVA))
    sin_suavizar = probabilidades(cuentas, alpha=0)
    print("NLL entrenamiento sin suavizar: %.6f"
          % perdida(sin_suavizar, ENTRENAMIENTO))
    print("NLL reserva sin suavizar:", perdida(sin_suavizar, RESERVA))
    rng = random.Random(7)
    for _ in range(10):
        texto, completo = generar(tabla, rng)
        estado = "fin" if completo else "truncado"
        print(repr(texto), estado)

In [12]:
ejecutar()

Columnas: ('a', 'b', 'l', 'n', '$')
^ [4, 0, 0, 0, 0]
a [0, 0, 2, 3, 3]
b [1, 0, 0, 0, 0]
l [1, 1, 0, 0, 0]
n [2, 0, 0, 1, 1]
NLL uniforme: 1.609438
NLL entrenamiento: 1.113961
NLL reserva: 1.409950
NLL entrenamiento sin suavizar: 0.747513
NLL reserva sin suavizar: inf
'ablanbananabbnall' fin
'bb' fin
'a' fin
'abalnannblaallbanlbn' truncado
'lanl' fin
'lb' fin
'annanannn' fin
'a' fin
'blb' fin
'' fin


## 7. Experimenta

Cambia la semilla y vuelve a ejecutar (ejercicio 9): ¿qué cambia y qué no? Después prueba otros valores de `alpha` (ejercicio 12) y observa la fila de `a` y la pérdida de entrenamiento. Escribe antes qué esperas que ocurra, y no uses la reserva para decidir.

In [13]:
semilla = 7  # prueba otras
rng = random.Random(semilla)
for _ in range(10):
    texto, completo = generar(tabla, rng)
    print(repr(texto), "fin" if completo else "truncado")

'ablanbananabbnall' fin
'bb' fin
'a' fin
'abalnannblaallbanlbn' truncado
'lanl' fin
'lb' fin
'annanannn' fin
'a' fin
'blb' fin
'' fin


In [14]:
alpha = 0.1  # prueba también 1 y 10
prueba = probabilidades(cuentas, alpha=alpha)
print("Fila de a:", [round(prueba["a"][s], 3) for s in SALIDAS])
print("Pérdida de entrenamiento:", round(perdida(prueba, ENTRENAMIENTO), 3))

Fila de a: [0.012, 0.012, 0.247, 0.365, 0.365]
Pérdida de entrenamiento: 0.817


## Ejercicios

1. Escribe las cinco transiciones de `alba`, incluyendo inicio y final. Explica por qué hay una más que letras.

2. Aplica la elección voraz a la tabla suavizada, empezando en el inicio y resolviendo los empates a favor de la salida que aparece antes en el orden `a`, `b`, `l`, `n`, final. ¿Qué cadena obtienes? ¿Termina?

3. Calcula, sin suavizado, la probabilidad de `l` después de `a` y de `a` después de `l`. Explica por qué no tienen que coincidir.

4. Con α=1, escribe las cinco probabilidades después de `n` y comprueba su suma.

5. Calcula la probabilidad suavizada de generar una cadena vacía. ¿Debe ocultarla el programa para evaluar honestamente las muestras?

6. Sigue el camino completo de `anna` con la tabla suavizada: calcula su probabilidad y su NLL media, incluyendo el final.

7. Con la tabla suavizada, calcula a mano la probabilidad del camino de `lana`. Identifica la transición que recibe probabilidad cero sin suavizado.

8. Justifica por qué el modelo uniforme tiene pérdida media ln 5 en cualquier cadena válida. ¿Cambiaría si usáramos logaritmos en base dos?

9. Cambia solo la semilla. Compara muestras y pérdidas, y explica qué parte del procedimiento ha cambiado.

10. Explica por qué no podemos evaluar `rosa` con esta tabla. Distingue letra desconocida de pareja no observada.

11. Compara el contexto utilizado después de `an` y `alan`. ¿Qué información necesitaría un modelo para diferenciarlos? (Lo hará el capítulo 4.)

12. Prueba `alpha=0.1` y `alpha=10`, en el cuaderno o cambiando la línea 72 del programa. Describe qué ocurre con la fila de `a` y con la pérdida de entrenamiento. No uses la reserva para decidir cuál es mejor.

13. Redacta una conclusión de cinco líneas sobre la comparación con el uniforme: menciona la reserva, el número de transiciones, la limitación de los datos y la diferencia entre puntuación y calidad de las muestras.

## Referencias

Idea docente: primera lección de [makemore](https://www.youtube.com/watch?v=PaCmpygFfXo), dentro de [Zero to Hero](https://github.com/karpathy/nn-zero-to-hero), de Andrej Karpathy. Colección mínima, código y explicaciones propios.